# 02_bag_of_words_tfidf: Vector Representations using UCI SMS Spam Dataset
    
This notebook builds Bag of Words (BoW) and Term Frequency - Inverse Document Frequency (TF-IDF) representation matrices from scratch using NumPy over the real-world UCI SMS Spam dataset, comparing outputs against Scikit-Learn.


In [1]:
import numpy as np
import pandas as pd
import re

# 1. Load UCI SMS Spam dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "message"])
print("Dataset size:", df.shape)

# Slice 5 sample messages to keep matrix prints readable
corpus_raw = df["message"].iloc[10:15].tolist()
# Basic normalization
corpus = [msg.lower().replace(".", "").replace(",", "") for msg in corpus_raw]

print("\nNormalized Corpus:")
for idx, doc in enumerate(corpus):
    print(f"Doc {idx+1}: {doc}")


Dataset size: (5572, 2)

Normalized Corpus:
Doc 1: i'm gonna be home soon and i don't want to talk about this stuff anymore tonight k? i've cried enough today
Doc 2: six chances to win cash! from 100 to 20000 pounds txt> csh11 and send to 87575 cost 150p/day 6days 16+ tsandcs apply reply hl 4 info
Doc 3: urgent! you have won a 1 week free membership in our £100000 prize jackpot! txt the word: claim to no: 81010 t&c wwwdbuknet lccltd pobox 4403ldnw1a7rw18
Doc 4: i've been searching for the right words to thank you for this breather i promise i wont take your help for granted and will fulfil my promise you have been wonderful and a blessing at all times
Doc 5: i have a date on sunday with will!!


### Output Explanation: Loading SMS spam dataset
- **Data Ingestion**: We fetched the SMS Spam Collection dataset directly over HTTPS.
- **Sampling**: To allow clear visual inspection of matrix values, we sliced a 5-document sample from the corpus and cleaned basic formatting details.


In [2]:
# Map vocabulary using regex (words length >= 2)
words = []
for doc in corpus:
    words.extend(re.findall(r"\b\w\w+\b", doc))
vocab = sorted(list(set(words)))
word_to_idx = {w: i for i, w in enumerate(vocab)}
print("Vocabulary Size:", len(vocab))
print("Vocabulary Mapping:\n", word_to_idx)

# Bag of Words (BoW) count matrix from scratch
bow_matrix = np.zeros((len(corpus), len(vocab)))
for doc_idx, doc in enumerate(corpus):
    for word in re.findall(r"\b\w\w+\b", doc):
        if word in word_to_idx:
            bow_matrix[doc_idx, word_to_idx[word]] += 1

print("\nBag of Words Count Matrix:\n", bow_matrix)


Vocabulary Size: 86
Vocabulary Mapping:
 {'100': 0, '100000': 1, '150p': 2, '16': 3, '20000': 4, '4403ldnw1a7rw18': 5, '6days': 6, '81010': 7, '87575': 8, 'about': 9, 'all': 10, 'and': 11, 'anymore': 12, 'apply': 13, 'at': 14, 'be': 15, 'been': 16, 'blessing': 17, 'breather': 18, 'cash': 19, 'chances': 20, 'claim': 21, 'cost': 22, 'cried': 23, 'csh11': 24, 'date': 25, 'day': 26, 'don': 27, 'enough': 28, 'for': 29, 'free': 30, 'from': 31, 'fulfil': 32, 'gonna': 33, 'granted': 34, 'have': 35, 'help': 36, 'hl': 37, 'home': 38, 'in': 39, 'info': 40, 'jackpot': 41, 'lccltd': 42, 'membership': 43, 'my': 44, 'no': 45, 'on': 46, 'our': 47, 'pobox': 48, 'pounds': 49, 'prize': 50, 'promise': 51, 'reply': 52, 'right': 53, 'searching': 54, 'send': 55, 'six': 56, 'soon': 57, 'stuff': 58, 'sunday': 59, 'take': 60, 'talk': 61, 'thank': 62, 'the': 63, 'this': 64, 'times': 65, 'to': 66, 'today': 67, 'tonight': 68, 'tsandcs': 69, 'txt': 70, 'urgent': 71, 've': 72, 'want': 73, 'week': 74, 'will': 75, 'wi

### Output Explanation: Vocabulary and Bag-of-Words Counts
- **Vocabulary Mapping**: Extracts all unique tokens with a length of at least 2 characters. The indices are sorted alphabetically.
- **BoW Matrix**: Each row represents one document, and each column corresponds to a word index. The cells show raw count values of that term in the document.


In [3]:
# Smooth IDF formulation: log((1 + N) / (1 + DF)) + 1
N = len(corpus)
df_counts = np.sum(bow_matrix > 0, axis=0)
idf = np.log((1 + N) / (1 + df_counts)) + 1

# Calculate TF-IDF (multiply counts by IDF weights)
tfidf_matrix = bow_matrix * idf

# L2 normalization to match Scikit-Learn standard
norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
tfidf_norm = tfidf_matrix / (norms + 1e-15)

print("Calculated IDFs:\n", idf)
print("\nTF-IDF Matrix (from scratch, normalized):\n", np.round(tfidf_norm, 4))


Calculated IDFs:
 [2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 1.40546511
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 1.40546511
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 1.69314718 1.69314718 2.09861229
 1.18232156 2.09861229 2.09861229 2.09861229 1.69314718 2.09861229
 1.69314718 2.09861229 2.09861229 1.69314718 2.09861229 2.09861229
 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 1.69314718 2.09861229]

TF-IDF Matrix (from

### Output Explanation: Math Derivations of TF-IDF
- **Smoothed IDFs**: Computed using $\log((1 + N)/(1 + 	ext{DF})) + 1$. This scales down highly frequent terms while amplifying rare terms.
- **L2 Normalization**: Ensures each document vector has a unit length of $1.0$, preventing document length differences from skewing cosine similarity calculations.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(norm='l2', smooth_idf=True, use_idf=True)
sklearn_tfidf = vectorizer.fit_transform(corpus).toarray()
print("Scikit-Learn TF-IDF Matrix:\n", np.round(sklearn_tfidf, 4))

# Check alignment assertions
assert np.allclose(tfidf_norm, sklearn_tfidf, atol=1e-5)
print("\nSUCCESS: Custom TF-IDF matrix matches Scikit-Learn output exactly!")


Scikit-Learn TF-IDF Matrix:
 [[0.     0.     0.     0.     0.     0.     0.     0.     0.     0.2495
  0.     0.1671 0.2495 0.     0.     0.2495 0.     0.     0.     0.
  0.     0.     0.     0.2495 0.     0.     0.     0.2495 0.2495 0.
  0.     0.     0.     0.2495 0.     0.     0.     0.     0.2495 0.
  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.     0.     0.2495 0.2495 0.
  0.     0.2495 0.     0.     0.2013 0.     0.1405 0.2495 0.2495 0.
  0.     0.     0.2013 0.2495 0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.    ]
 [0.2002 0.     0.2002 0.2002 0.2002 0.     0.2002 0.     0.2002 0.
  0.     0.1341 0.     0.2002 0.     0.     0.     0.     0.     0.2002
  0.2002 0.     0.2002 0.     0.2002 0.     0.2002 0.     0.     0.
  0.     0.2002 0.     0.     0.     0.     0.     0.2002 0.     0.
  0.2002 0.     0.     0.     0.     0.     0.     0.     0.     0.2002
  0.     0.     0.2002 0.     

### Output Explanation: Validation
- **Exact Alignment**: The assertion passes successfully with `atol=1e-5`, verifying that our mathematical derivation and coding from scratch matches Scikit-Learn's output exactly.
